# Preprocessing

## Load useful libs

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from hypyp import analyses
%pip install mne-qt-browser
%pip install PySide6
mne.viz.set_browser_backend('qt')

/Users/felix/Documents/MPI/EEG_hyperscanning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached pyside6-6.11.2-cp310-abi3-macosx_13_0_universal2.whl (571 kB)
  Using cached pyside6_addons-6.11.2-cp310-abi3-macosx_13_0_universal2.whl (332.0 MB)

[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Using qt as 2D backend.


## Get Raw data

In [2]:
raw = mne.io.read_raw_brainvision(
    "data/ds007471/sub-01/eeg/sub-01_task-jointaction_eeg.vhdr",
    preload=True,
)

print(raw.info)
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"Duration: {raw.times[-1]:.1f}")

Extracting parameters from data/ds007471/sub-01/eeg/sub-01_task-jointaction_eeg.vhdr...


Setting channel info structure...


/var/folders/2x/v_ydgf450nd23wnb3_hcjtw40000gn/T/ipykernel_3307/690757239.py:1: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  raw = mne.io.read_raw_brainvision(


Reading 0 ... 2318619  =      0.000 ...  2318.619 secs...
<Info | 7 non-empty values
 bads: []
 ch_names: VEOG_b_R, VEOG_a_R, F7_R, F3_R, Fz_R, F4_R, F8_R, FC5_R, FC1_R, ...
 chs: 64 EEG
 custom_ref_applied: False
 highpass: 1.0 Hz
 lowpass: 125.0 Hz
 meas_date: 2022-11-30 14:50:15 UTC
 nchan: 64
 projs: []
 sfreq: 1000.0 Hz
>
Sampling rate: 1000.0 Hz
Duration: 2318.6


**Raw Data:**
- Number of Channels: 64 Channels simultaneously (32 per person)
- Two eye-movement-channels: `VEOG_b_R` and `VEOG_a_R` 
- Sampling rate: 1000 Hz
- Duration: 2,3186 seconds ≈ 38.6 minutes
- Frequency Range 1-125 Hz

## Split into two participants and drop non-brain channels

In [3]:
# VEOG_b, VEOG_a, HEOG_l, HEOG_r are eye-movement electrodes, not brain signal, and ref_lm is the linked-mastoid reference
non_eeg = {"VEOG_b", "VEOG_a", "HEOG_l", "HEOG_r", "ref_lm"}

eeg_R = [ch for ch in raw.ch_names if ch.endswith("_R") and ch[:-2] not in non_eeg]
eeg_L = [ch for ch in raw.ch_names if ch.endswith("_L") and ch[:-2] not in non_eeg]

raw_R = raw.copy().pick(eeg_R)
raw_L = raw.copy().pick(eeg_L)

raw_R.rename_channels({name: name.removesuffix("_R") for name in raw_R.ch_names})
raw_L.rename_channels({name: name.removesuffix("_L") for name in raw_L.ch_names})

print(raw_L)

<RawBrainVision | sub-01_task-jointaction_eeg.eeg, 27 x 2318620 (2318.6 s), ~477.6 MiB, data loaded>


## Set Montage

In [4]:
raw_R.set_montage("standard_1020", on_missing="warn")
raw_L.set_montage("standard_1020", on_missing="warn")

# Give MNE Information on event-markers
events, event_id = mne.events_from_annotations(raw)

# Look up the numeric code for marker Sn, e.g. event_code(105) for S105
def event_code(n):
    return event_id[f"Stimulus/S{n:>3}"]

print(f"{len(events)} total markers")
print(f"Event types: {event_id}")

Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  3'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S 10'), np.str_('Stimulus/S 11'), np.str_('Stimulus/S105'), np.str_('Stimulus/S106'), np.str_('Stimulus/S107'), np.str_('Stimulus/S128'), np.str_('Stimulus/S130'), np.str_('Stimulus/S131'), np.str_('Stimulus/S132')]
2229 total markers
Event types: {np.str_('Stimulus/S  1'): 1, np.str_('Stimulus/S  2'): 2, np.str_('Stimulus/S  3'): 3, np.str_('Stimulus/S  4'): 4, np.str_('Stimulus/S 10'): 10, np.str_('Stimulus/S 11'): 11, np.str_('Stimulus/S105'): 105, np.str_('Stimulus/S106'): 106, np.str_('Stimulus/S107'): 107, np.str_('Stimulus/S128'): 128, np.str_('Stimulus/S130'): 130, np.str_('Stimulus/S131'): 131, np.str_('Stimulus/S132'): 132}


## Filter out practice trials and label each real trial's condition

In [5]:
s1_sample = events[events[:, 2] == event_code(1)][0, 0]

all_trial_events = events[events[:, 2] == event_code(105)]
real_trial_events = all_trial_events[all_trial_events[:, 0] > s1_sample]

print(f"{len(all_trial_events)} total S105 markers "
      f"({len(all_trial_events) - len(real_trial_events)} practice, {len(real_trial_events)} real test trials)")

s10, s11 = event_code(10), event_code(11)
condition_events = events[np.isin(events[:, 2], [s10, s11])]

trial_condition = []
for sample in real_trial_events[:, 0]:
    preceding = condition_events[condition_events[:, 0] < sample]
    trial_condition.append("duet" if preceding[-1, 2] == s10 else "constant_pitch")
trial_condition = np.array(trial_condition)

print(f"Duet trials: {(trial_condition == 'duet').sum()}, "
      f"constant-pitch trials: {(trial_condition == 'constant_pitch').sum()}")

63 total S105 markers (6 practice, 57 real test trials)
Duet trials: 28, constant-pitch trials: 29


## Epoch the trials

In [6]:
tmin, tmax = -0.5, 5.0  # seconds relative to trial start

epochs_R = mne.Epochs(
    raw_R, real_trial_events,
    event_id={"trial_start": event_code(105)},
    tmin=tmin, tmax=tmax, baseline=(tmin, 0),
    preload=True,
)
epochs_L = mne.Epochs(
    raw_L, real_trial_events,
    event_id={"trial_start": event_code(105)},
    tmin=tmin, tmax=tmax, baseline=(tmin, 0),
    preload=True,
)

print(f"{len(epochs_R)} trials x {len(epochs_R.ch_names)} channels x {epochs_R.get_data().shape[-1]} timepoints")
assert len(epochs_R) == len(epochs_L) == len(trial_condition), "trial counts should all match"

Not setting metadata
57 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 57 events and 5501 original time points ...
0 bad epochs dropped
Not setting metadata
57 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 57 events and 5501 original time points ...
0 bad epochs dropped
57 trials x 27 channels x 5501 timepoints


In [7]:
epochs_R.info
epochs_R.plot(scalings=dict(eeg=100e-6), n_epochs=5, n_channels=20)


<mne_qt_browser._pg_figure.MNEQtBrowser(0x16409cef0) at 0x17ea2cdc0>

Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
none


/Users/felix/Documents/MPI/EEG_hyperscanning/.venv/lib/python3.11/site-packages/mne_qt_browser/_utils.py:36: RuntimeWarning: libpyside: Failed to disconnect (None) from signal "triggered()".
  sig.disconnect()
/Users/felix/Documents/MPI/EEG_hyperscanning/.venv/lib/python3.11/site-packages/mne_qt_browser/_utils.py:36: RuntimeWarning: libpyside: Failed to disconnect (None) from signal "triggered()".
  sig.disconnect()
/Users/felix/Documents/MPI/EEG_hyperscanning/.venv/lib/python3.11/site-packages/mne_qt_browser/_utils.py:36: RuntimeWarning: libpyside: Failed to disconnect (None) from signal "triggered()".
  sig.disconnect()
/Users/felix/Documents/MPI/EEG_hyperscanning/.venv/lib/python3.11/site-packages/mne_qt_browser/_utils.py:36: RuntimeWarning: libpyside: Failed to disconnect (None) from signal "triggered()".
  sig.disconnect()
/Users/felix/Documents/MPI/EEG_hyperscanning/.venv/lib/python3.11/site-packages/mne_qt_browser/_utils.py:36: RuntimeWarning: libpyside: Failed to disconnect (No

## Load behavioral file

In [7]:
path = "data/ds007471/sub-01/beh/sub-01_task-jointaction_beh.tsv"
pd.read_csv(path, sep=r"\s+")

,PairNumber,ParticipantNumber,BlockNumber,TrialNumber,ExperimentalCondition,PartPerformed,ToneSequence,JointAgencyRatings,MeanSynchronizationPerformance,SDSynchronizationPerformance
0,1,11,1,1,1,0,1,6,0.60603,0.282770
1,1,11,1,2,1,0,1,5,0.14658,0.157710
2,1,11,1,3,1,0,1,5,0.43527,0.257650
3,1,11,1,4,1,0,1,5,0.51752,0.367430
4,1,11,1,5,1,0,1,4,0.67171,0.350170
...,...,...,...,...,...,...,...,...,...,...
75,1,12,8,36,0,1,7,6,0.33786,0.175900
76,1,12,8,37,0,1,7,6,0.20117,0.126080
77,1,12,8,38,0,1,7,6,0.27754,0.235970
78,1,12,8,39,0,1,7,6,0.14826,0.064079


## Save the epochs for further use

In [9]:
epochs_R.save("data/ds007471/derivatives/sub-01_R-epo.fif", overwrite=True)
epochs_L.save("data/ds007471/derivatives/sub-01_L-epo.fif", overwrite=True)

[PosixPath('/Users/felix/Documents/MPI/EEG_hyperscanning/data/ds007471/derivatives/sub-01_L-epo.fif')]